In [1]:
%matplotlib inline
import pandas as pd
import numpy as np
import matplotlib
# matplotlib.use('agg')
import matplotlib.pyplot as plt
import os
# from tqdm.notebook import  tqdm
from tqdm import  tqdm
import talib
import datetime
import math
import  mplfinance as mpf

import sys
sys.path.append('../../DataSource/同花顺')
import datasource
sys.path.append('../..')
import Utils
# 这个是筛选多少天上涨多少的，
codes = datasource.get_codes()
print(f'股票数量:{len(codes)},第一支股票:{codes[0]}')
code = codes[0]
dt = datasource.getData(code)
dt.head()

股票数量:6974,第一支股票:000001


,Id,Code,Date,Open,High,Low,Close,Amount,Volume
0,1,000001,2014-01-02,12.12,12.30,12.05,12.23,596223740.0,48991089.0
1,2,000001,2014-01-03,12.15,12.16,11.78,11.93,656631300.0,55111484.0
2,3,000001,2014-01-06,11.89,12.00,11.50,11.67,679280380.0,58211823.0
3,4,000001,2014-01-07,11.53,11.76,11.51,11.63,393977580.0,33840749.0
4,5,000001,2014-01-08,11.64,11.95,11.53,11.76,538436170.0,45776816.0


In [2]:
# 先看看涨幅
dt['preClose'] = dt['Close'].shift() # 往下移动，
dt['upRate'] = (dt['Close']-dt['preClose'])/dt['preClose']*100 # 今天的涨幅
for i in [1,3,5]:
    dt[f'nextRate{i}'] = (dt['Close'].shift(-i)-dt['Close'])/dt['Close']*100 # 未来几天的涨幅
dt2 = dt.loc[dt['upRate']>9.5, :]
dt2.head()

,Id,Code,Date,Open,High,Low,Close,Amount,Volume,preClose,upRate,nextRate1,nextRate3,nextRate5
220,221,000001,2014-11-28,11.27,12.44,11.18,12.44,5.793392e+09,484483930.0,11.31,9.991158,-2.009646,5.305466,16.800643
307,308,000001,2015-04-10,18.00,19.80,17.85,19.80,6.339649e+09,334020970.0,18.00,10.000000,-16.464646,-15.909091,-14.494949
1492,1493,000001,2020-07-06,14.60,15.68,14.59,15.68,7.168653e+09,471146080.0,14.25,10.035088,-1.275510,-0.956633,-5.038265
1932,1933,000001,2022-11-29,12.16,12.99,12.13,12.99,6.026007e+09,474927650.0,11.81,9.991533,0.307929,-0.692841,3.387221
2228,2229,000001,2024-02-21,9.78,10.80,9.77,10.80,5.295259e+09,505528460.0,9.82,9.979633,0.925926,-2.500000,-2.870370


In [3]:
# 这里看一下结果
for i in [1,3,5]:
    j = dt2[f'nextRate{i}']
    print(f'未来{i}天平均涨幅:{j.mean():.2f},标准差:{j.std():.2f},最大涨幅:{j.max():.2f},最小涨幅:{j.min():.2f},')

未来1天平均涨幅:-3.70,标准差:7.23,最大涨幅:0.93,最小涨幅:-16.46,
未来3天平均涨幅:-2.95,标准差:7.83,最大涨幅:5.31,最小涨幅:-15.91,
未来5天平均涨幅:-0.44,标准差:11.58,最大涨幅:16.80,最小涨幅:-14.49,


In [10]:
# 这里是全部股票
dts = []
for i in tqdm(range(len(codes))):
    code = codes[i]
    if code.startswith('00') or code.startswith('60'):
        dt = datasource.getData(code, start_date='2025-01-01')
        dt['preClose'] = dt['Close'].shift() # 往下移动，
        dt['upRate'] = (dt['Close']-dt['preClose'])/dt['preClose']*100 # 今天的涨幅
        for i in [1,3,5]:
            dt[f'nextRate{i}'] = (dt['Close'].shift(-i)-dt['Close'])/dt['Close']*100 # 未来几天的涨幅
        dt2 = dt.loc[(dt['upRate']>9.5) & (dt['upRate']< 10.5), :]
        if len(dt2)>0:
            dts.append(dt2)

dt_all = pd.concat(dts)   

100%|█████████████████████████████████████████████████████████████████████████████| 6974/6974 [00:28<00:00, 247.07it/s]


In [11]:
for i in [1,3,5]:
    j = dt_all[f'nextRate{i}']
    print(f'未来{i}天平均涨幅:{j.mean():.2f},标准差:{j.std():.2f},最大涨幅:{j.max():.2f},最小涨幅:{j.min():.2f},')

未来1天平均涨幅:1.78,标准差:6.00,最大涨幅:14.05,最小涨幅:-11.27,
未来3天平均涨幅:2.02,标准差:11.82,最大涨幅:33.67,最小涨幅:-27.17,
未来5天平均涨幅:1.53,标准差:14.83,最大涨幅:61.66,最小涨幅:-40.95,
